In [ ]:
# Example 1: Inspect the package's public API surface.
import genai_prices

public_names = [n for n in dir(genai_prices) if not n.startswith('_')]
print(f"Public names ({len(public_names)}): {public_names[:30]}")

# Verify at least one of the documented top-level helpers exists.
candidates = ['calc_price', 'calc_price_sync', 'data', 'types', 'extract_usage']
present = [c for c in candidates if hasattr(genai_prices, c)]
print(f"Present API candidates: {present}")
assert len(present) >= 1, f"Expected at least one of {candidates} in genai_prices"


In [ ]:
# Example 2: Explore the bundled pricing data module.
from genai_prices import data

data_names = [n for n in dir(data) if not n.startswith('_')]
print(f"genai_prices.data exposes {len(data_names)} names")
print(f"First few: {data_names[:15]}")

# The data module typically has a 'providers' collection of provider price tables.
if hasattr(data, 'providers'):
    providers = data.providers
    print(f"providers type: {type(providers).__name__}")
    try:
        n = len(providers)
        print(f"providers length: {n}")
        assert n > 0, "Expected at least one provider in pricing data"
    except TypeError:
        # Not sized; just ensure it's truthy.
        assert providers is not None


In [ ]:
# Example 3: Exercise the types/models submodule (Pydantic models).
import importlib

# The Pydantic/dataclass types live in genai_prices.types (measured: there is no genai_prices.models); import it directly
import genai_prices.types as type_module
print(f"Loaded {type_module.__name__}")

# List class-like attributes (Pydantic models / dataclasses).
classes = [n for n in dir(type_module) if not n.startswith('_') and isinstance(getattr(type_module, n), type)]
print(f"Found {len(classes)} class names; sample: {classes[:10]}")
assert len(classes) >= 1
assert {"ModelPrice", "Usage", "Provider"} <= set(classes), classes


In [ ]:
# Example 4: Call calc_price (the package's headline API) with a small token usage payload.
# calc_price takes a Usage object (attribute-based), not a plain dict.
from genai_prices import Usage, calc_price

usage = Usage(input_tokens=1000, output_tokens=500)
result = None
last_err = None
for model_ref in ('gpt-4o', 'gpt-4o-mini', 'gpt-3.5-turbo', 'claude-3-5-sonnet-latest'):
    try:
        result = calc_price(usage, model_ref)
        print(f"calc_price succeeded for model_ref={model_ref!r}")
        break
    except Exception as e:
        last_err = e
        continue

assert result is not None, f"calc_price failed for all probe models; last error: {last_err!r}"
print(f"result type: {type(result).__name__}")
print(f"result repr (truncated): {repr(result)[:300]}")


In [ ]:
# Example 5: Verify the calc_price result exposes a numeric total cost field.
from decimal import Decimal

from genai_prices import Usage, calc_price

usage = Usage(input_tokens=1000, output_tokens=500)
result = None
for model_ref in ('gpt-4o', 'gpt-4o-mini', 'gpt-3.5-turbo'):
    try:
        result = calc_price(usage, model_ref)
        break
    except Exception:
        continue

assert result is not None, "calc_price did not succeed for any probe model"

# Look for any numeric cost-like attribute.
cost_fields = [n for n in dir(result) if 'price' in n.lower() or 'cost' in n.lower() or 'total' in n.lower()]
print(f"Candidate cost fields: {cost_fields}")

found_numeric = False
for f in cost_fields:
    if f.startswith('_'):
        continue
    v = getattr(result, f, None)
    if isinstance(v, (int, float, Decimal)):
        print(f"  {f} = {v}")
        found_numeric = True

assert found_numeric, f"Expected at least one numeric cost field on result; got fields {cost_fields}"
print("genai_prices smoke test passed")
